# Playground

In [ ]:
import numpy as np
import pandas as pd
from collections import Counter
import json

In [18]:
raw_data_customers_path = "../../data/raw/customers.json"
raw_data_products_path = "../../data/raw/products.csv"
raw_data_sales_path = "../../data/raw/sales_data.json"

In [19]:
customers_df = pd.read_json(raw_data_customers_path)
products_df = pd.read_csv(raw_data_products_path)
#sales_df = pd.read_json(raw_data_sales_path) // Error, maybe be related to parsing

In [20]:
customers_df.head(5)

,CustomerID,Name,Region,SignUpDate,Email,LoyaltyPoints
0,8220,Brandon Bender,North,2023-08-14,mirandajoel@example.org,129
1,6670,Kari Ford,North,2020-03-02,brucewesley@example.org,528
2,9865,Andrea Leonard,NaN,2020-11-26,newmantimothy@example.com,2182
3,5082,Christina Mcdowell,NaN,2022-12-07,smithdouglas@example.com,3091
4,3737,Kevin Martin,South,2019-11-02,pinedakelli@example.net,1156


In [21]:
products_df.head(5)

,ProductID,ProductName,Category,Price,Supplier
0,207,,Puzzle,64,Brown-Garcia
1,732,Joseph,RC Toy,170,Ortiz Inc
2,463,Jacob,RC Toy,108,Hinton-Patterson
3,163,Shawn,Action Figure,107,"Rios, Cannon and Wheeler"
4,255,Daniel,Puzzle,163,Summers LLC


In [22]:
with open(raw_data_sales_path) as f:
    sales_raw_data = json.load(f)

print(type(sales_raw_data), len(sales_raw_data))
print(Counter(type(r).__name__ for r in sales_raw_data))

<class 'list'> 2983
Counter({'dict': 2895, 'str': 88})


In [23]:
bad = [(i, r) for i, r in enumerate(sales_raw_data) if not isinstance(r, dict)]
print(len(bad))
for i, r in bad[:5]:
    print(i, repr(r)[:200])

88
36 '{"SaleID": 423, "ProductID": 115, "CustomerID": 2318, "Quantity": 10, "TotalAmount": 910, "SaleDate": "2021-11-12"}'
120 '{"SaleID": 6461, "ProductID": 335, "CustomerID": 9926, "Quantity": 2, "TotalAmount": 24, "SaleDate": "2021-03-20"}'
185 '{"SaleID": 4264, "ProductID": 848, "CustomerID": 9898, "Quantity": 1, "TotalAmount": 174, "SaleDate": "2021-09-23"}'
228 '{"SaleID": 6742, "ProductID": 607, "CustomerID": 4756, "Quantity": 4, "TotalAmount": 732, "SaleDate": "2021-01-29"}'
311 '{"SaleID": 2730, "ProductID": 301, "CustomerID": 8124, "Quantity": 4, "TotalAmount": 736, "SaleDate": "2022-03-05"}'


88 elements are double serialized

In [24]:
records, malformed = [], []

for i, r in enumerate(sales_raw_data):
    if isinstance(r, dict):
        records.append(r)
    elif isinstance(r, str):
        try:
            records.append(json.loads(r))
        except json.JSONDecodeError:
            malformed.append((i, r))
    else:
        malformed.append((i, r))

sales_df = pd.json_normalize(records)

In [25]:
sales_df[sales_df.SaleID == 423] # There are elements in the json that contain other jsons... error that has to be solved

,SaleID,ProductID,CustomerID,Quantity,TotalAmount,SaleDate,find.SaleID,find.ProductID,find.CustomerID,find.Quantity,...,heavy.CustomerID,heavy.Quantity,heavy.TotalAmount,heavy.SaleDate,public.SaleID,public.ProductID,public.CustomerID,public.Quantity,public.TotalAmount,public.SaleDate
36,423.0,115.0,2318.0,10.0,910.0,2021-11-12,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [26]:
Counter(tuple(r) for r in sales_raw_data if isinstance(r, dict)) # 98 elements nested, generating malformed data.

Counter({('SaleID',
          'ProductID',
          'CustomerID',
          'Quantity',
          'TotalAmount',
          'SaleDate'): 2797,
         ('find',): 2,
         ('country',): 2,
         ('change',): 2,
         ('miss',): 1,
         ('ability',): 1,
         ('small',): 1,
         ('feeling',): 1,
         ('person',): 1,
         ('article',): 1,
         ('up',): 1,
         ('condition',): 1,
         ('bring',): 1,
         ('spend',): 1,
         ('course',): 1,
         ('simple',): 1,
         ('tonight',): 1,
         ('eat',): 1,
         ('member',): 1,
         ('standard',): 1,
         ('our',): 1,
         ('recently',): 1,
         ('attention',): 1,
         ('culture',): 1,
         ('summer',): 1,
         ('audience',): 1,
         ('seek',): 1,
         ('memory',): 1,
         ('from',): 1,
         ('keep',): 1,
         ('hair',): 1,
         ('response',): 1,
         ('daughter',): 1,
         ('civil',): 1,
         ('cause',): 1,
         ('m

In [27]:
sales_df.keys()

Index(['SaleID', 'ProductID', 'CustomerID', 'Quantity', 'TotalAmount',
       'SaleDate', 'find.SaleID', 'find.ProductID', 'find.CustomerID',
       'find.Quantity',
       ...
       'heavy.CustomerID', 'heavy.Quantity', 'heavy.TotalAmount',
       'heavy.SaleDate', 'public.SaleID', 'public.ProductID',
       'public.CustomerID', 'public.Quantity', 'public.TotalAmount',
       'public.SaleDate'],
      dtype='str', length=556)

In [28]:
records, malformed = [], []
for i, r in enumerate(sales_raw_data):
    if isinstance(r, str):
        try:
            r = json.loads(r)
        except json.JSONDecodeError:
            malformed.append((i, r)); continue

    if isinstance(r, dict):
        if len(r) == 1 and isinstance(next(iter(r.values())), dict):
            r = next(iter(r.values()))
        records.append(r)
    else:
        malformed.append((i, r))

sales_df = pd.DataFrame(records)

In [29]:
sales_df

,SaleID,ProductID,CustomerID,Quantity,TotalAmount,SaleDate,condition,standard,policy,example,a
0,5083.0,343.0,9320.0,9.0,1431.0,2023-06-19,NaN,NaN,NaN,NaN,NaN
1,6690.0,463.0,8458.0,6.0,648.0,2020-05-15,NaN,NaN,NaN,NaN,NaN
2,8362.0,750.0,8226.0,1.0,172.0,2022-11-03,NaN,NaN,NaN,NaN,NaN
3,6412.0,419.0,3217.0,9.0,1665.0,2023-06-17,NaN,NaN,NaN,NaN,NaN
4,1721.0,840.0,7811.0,8.0,800.0,2022-04-21,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
2978,27.0,789.0,6886.0,9.0,612.0,2021-01-09,NaN,NaN,NaN,NaN,NaN
2979,3931.0,676.0,3284.0,1.0,62.0,2020-04-22,NaN,NaN,NaN,NaN,NaN
2980,9668.0,198.0,3064.0,2.0,28.0,2020-10-11,NaN,NaN,NaN,NaN,NaN
2981,1634.0,659.0,3116.0,7.0,413.0,2022-02-04,NaN,NaN,NaN,NaN,NaN


In [30]:
bronze = pd.DataFrame([
    {
        "_row_num": i,
        "_source_file": "sales_data.json",
        "_ingested_at": pd.Timestamp.utcnow(),
        "_payload": json.dumps(item, ensure_ascii=False),
    }
    for i, item in enumerate(sales_raw_data)
])

/tmp/ipykernel_116278/1119639899.py:5: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  "_ingested_at": pd.Timestamp.utcnow(),


In [31]:
bronze

,_row_num,_source_file,_ingested_at,_payload
0,0,sales_data.json,2026-08-16 17:22:38.367704+00:00,"{""SaleID"": 5083, ""ProductID"": 343, ""CustomerID..."
1,1,sales_data.json,2026-08-16 17:22:38.367762+00:00,"{""SaleID"": 6690, ""ProductID"": 463, ""CustomerID..."
2,2,sales_data.json,2026-08-16 17:22:38.367783+00:00,"{""SaleID"": 8362, ""ProductID"": 750, ""CustomerID..."
3,3,sales_data.json,2026-08-16 17:22:38.367798+00:00,"{""SaleID"": 6412, ""ProductID"": 419, ""CustomerID..."
4,4,sales_data.json,2026-08-16 17:22:38.367812+00:00,"{""SaleID"": 1721, ""ProductID"": 840, ""CustomerID..."
...,...,...,...,...
2978,2978,sales_data.json,2026-08-16 17:22:38.408243+00:00,"{""SaleID"": 27, ""ProductID"": 789, ""CustomerID"":..."
2979,2979,sales_data.json,2026-08-16 17:22:38.408254+00:00,"{""SaleID"": 3931, ""ProductID"": 676, ""CustomerID..."
2980,2980,sales_data.json,2026-08-16 17:22:38.408265+00:00,"{""SaleID"": 9668, ""ProductID"": 198, ""CustomerID..."
2981,2981,sales_data.json,2026-08-16 17:22:38.408276+00:00,"{""SaleID"": 1634, ""ProductID"": 659, ""CustomerID..."
